<a href="https://colab.research.google.com/github/Kamali1265/ML-Ops-Tasks/blob/main/DAY-6/model_traffic_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h2 style='color:blue' align="center">Traffic Analysis and Prediction</h2>

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings("ignore")

In [15]:
df = pd.read_csv("indian_roads_dataset.csv")

In [16]:
df.head()

,accident_id,city,state,latitude,longitude,date,time,hour,day_of_week,is_weekend,...,visibility,temperature,traffic_density,cause,accident_severity,vehicles_involved,casualties,is_peak_hour,festival,risk_score
0,0,Pune,Maharashtra,18.680827,73.930388,2023-10-22,5:00,5,Sunday,1,...,low,32,high,weather,fatal,2,2,0,NaN,0.85
1,1,Mumbai,Maharashtra,18.817732,72.790846,2023-05-21,4:00,4,Sunday,1,...,high,34,low,weather,major,4,3,0,NaN,0.10
2,2,Mumbai,Maharashtra,19.096889,72.819424,2024-07-10,13:00,13,Wednesday,0,...,low,21,medium,weather,minor,1,1,0,NaN,0.45
3,3,Chandigarh,Punjab,30.787805,76.847507,2025-03-30,11:00,11,Sunday,1,...,low,30,high,distraction,minor,5,2,0,NaN,0.65
4,4,Chennai,Tamil Nadu,12.965155,80.283313,2024-01-25,16:00,16,Thursday,0,...,high,24,low,distraction,minor,2,1,0,NaN,0.10


In [17]:
print(df.shape)

(20000, 24)


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   accident_id        20000 non-null  int64  
 1   city               20000 non-null  object 
 2   state              20000 non-null  object 
 3   latitude           20000 non-null  float64
 4   longitude          20000 non-null  float64
 5   date               20000 non-null  object 
 6   time               20000 non-null  object 
 7   hour               20000 non-null  int64  
 8   day_of_week        20000 non-null  object 
 9   is_weekend         20000 non-null  int64  
 10  road_type          20000 non-null  object 
 11  lanes              20000 non-null  int64  
 12  traffic_signal     20000 non-null  int64  
 13  weather            20000 non-null  object 
 14  visibility         20000 non-null  object 
 15  temperature        20000 non-null  int64  
 16  traffic_density    200

In [19]:
df.isnull().sum()

,0
accident_id,0
city,0
state,0
latitude,0
longitude,0
date,0
time,0
hour,0
day_of_week,0
is_weekend,0


In [20]:
df.drop("festival",axis=1,inplace=True)

In [21]:
df.drop(
    [
        "accident_id",
        "date",
        "time"
    ],
    axis=1,
    inplace=True
)

In [22]:
df.columns

Index(['city', 'state', 'latitude', 'longitude', 'hour', 'day_of_week',
       'is_weekend', 'road_type', 'lanes', 'traffic_signal', 'weather',
       'visibility', 'temperature', 'traffic_density', 'cause',
       'accident_severity', 'vehicles_involved', 'casualties', 'is_peak_hour',
       'risk_score'],
      dtype='object')

In [23]:
X = df.drop("risk_score",axis=1)

y = df["risk_score"]

In [24]:
X = pd.get_dummies(
    X,
    drop_first=True
)

In [25]:
X.shape

(20000, 43)

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [27]:
print(X_train.shape)
print(X_test.shape)

(16000, 43)
(4000, 43)


Modal Comparison

In [28]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Extra Trees": ExtraTreesRegressor(random_state=42)
}

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    r2 = r2_score(y_test, pred)

    results.append([name, r2])

comparison_df = pd.DataFrame(
    results,
    columns=["Model", "R2 Score"]
)

comparison_df.sort_values(
    by="R2 Score",
    ascending=False
)

,Model,R2 Score
0,Linear Regression,0.998506
3,Gradient Boosting,0.998383
2,Random Forest,0.998037
4,Extra Trees,0.997895
1,Decision Tree,0.996609


In [29]:
comparison_df.sort_values(
    by="R2 Score",
    ascending=False
)

,Model,R2 Score
0,Linear Regression,0.998506
3,Gradient Boosting,0.998383
2,Random Forest,0.998037
4,Extra Trees,0.997895
1,Decision Tree,0.996609


In [30]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    LinearRegression(),
    X,
    y,
    cv=5
)

scores

array([0.99880527, 0.99904131, 0.99845172, 0.99895565, 0.99879579])

In [31]:
scores.mean()

np.float64(0.998809947681606)

In [32]:
best_model = LinearRegression()

best_model.fit(X_train, y_train)

LinearRegression()

In [33]:
best_model.score(X_test, y_test)

0.9985060193328493

In [34]:
def predict_risk(input_data):

    input_df = pd.DataFrame(
        [input_data],
        columns=X.columns
    )

    prediction = best_model.predict(input_df)

    return prediction[0]

In [35]:
sample_input = X.iloc[0].values

predict_risk(sample_input)

np.float64(0.8506523567471798)

In [36]:
import pickle

with open(
    "traffic_risk_model.pkl",
    "wb"
) as f:

    pickle.dump(
        best_model,
        f
    )

In [37]:
import json

columns = {
    "data_columns": [col.lower() for col in X.columns]
}

with open(
    "columns.json",
    "w"
) as f:

    f.write(
        json.dumps(columns)
    )

In [38]:
import os

os.listdir()

['.config',
 'traffic_risk_model.pkl',
 'columns.json',
 'indian_roads_dataset.csv',
 'sample_data']

In [39]:
from google.colab import files

files.download("traffic_risk_model.pkl")
files.download("columns.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>